In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import get_isc_catalog, catalog_to_dataframe
from tqdm.auto import tqdm
from pathlib import Path
from torch.utils.data import Dataset
from typing import Optional, Union
class MultiCatalogDataset(Dataset):
    def __init__(self, root_dir : Union[str, Path], in_memory : bool = True, event_id_name : str = 'event_id', time_name : str = 'time', latitude_name : str = 'latitude', longitude_name : str = 'longitude', depth_name : str = 'depth', magnitude_name : str = 'magnitude', mag_root_dir : Optional[Union[str, Path]] = None, magnitude_type : str = 'magnitude_type', magnitude_family : str = "magnitude_family"):
        self.in_memory = in_memory
        self.event_id_name = event_id_name
        self.time_name = time_name
        self.latitude_name = latitude_name
        self.longitude_name = longitude_name
        self.depth_name = depth_name
        self.magnitude_name = magnitude_name
        self.magnitude_type = magnitude_type
        self.magnitude_family = magnitude_family
        self.root_dir = Path(root_dir) if isinstance(root_dir, str) else root_dir
        if mag_root_dir is not None:
            self.mag_root_dir = Path(mag_root_dir) if isinstance(mag_root_dir, str) else mag_root_dir
        else:
            self.mag_root_dir = None
        self.catalog_files = list(self.root_dir.glob("*.csv"))
        if self.mag_root_dir is not None:
            self.magnitude_files = list(Path(mag_root_dir).glob("*.csv")) if mag_root_dir is not None else []
        else:
            self.magnitude_files = []


        if self.in_memory:
            self.loaded_catalogs = {}
        else:
            self.loaded_catalogs = None
        self.cat_metadatas = []
        for file in self.catalog_files:
            if not file.is_file():
                continue
            file_key = file.stem
            df = pd.read_csv(file, parse_dates=[self.time_name])
            #df.sort_values(by=self.time_name, inplace=True)
            min_time = df[self.time_name].min().to_numpy()
            max_time = df[self.time_name].max().to_numpy()
            min_lat = df[self.latitude_name].min()
            max_lat = df[self.latitude_name].max()
            min_lon = df[self.longitude_name].min()
            max_lon = df[self.longitude_name].max()
            mean_mag = df[self.magnitude_name].mean()
            max_mag = df[self.magnitude_name].max()
            num_events = len(df)

            if self.in_memory:
                self.loaded_catalogs[file_key] = df
            
            self.cat_metadatas.append({"name" : file_key,
                "min_time": min_time,
                "max_time": max_time,
                "min_lat": min_lat,
                "max_lat": max_lat,
                "min_lon": min_lon,
                "max_lon": max_lon,
                "mean_mag" : mean_mag,
                "max_mag" : max_mag,
                "num_events" : num_events
            })
        self.cat_metadatas = pd.DataFrame(self.cat_metadatas)
        
    def _get_relevant_keys(self, start_time : Union[str, np.datetime64], end_time : Union[str, np.datetime64],
                           min_lat : float, max_lat : float,
                           min_lon : float, max_lon : float):
        if isinstance(start_time, str):
            start_time = np.datetime64(start_time)
        if isinstance(end_time, str):
            end_time = np.datetime64(end_time)
        relevant_df = self.cat_metadatas[(self.cat_metadatas['min_time'] >= start_time) & (self.cat_metadatas['max_time'] < end_time)]
        relevant_df = relevant_df[(relevant_df['min_lat'] >= min_lat) & (relevant_df['max_lat'] <= max_lat)]
        relevant_df = relevant_df[(relevant_df['min_lon'] >= min_lon) & (relevant_df['max_lon'] < max_lon)]
        return relevant_df['name'].tolist()

    def _load_data(self, start_time : Union[str, np.datetime64], end_time : Union[str, np.datetime64],
                           min_lat : float, max_lat : float,
                           min_lon : float, max_lon : float):
        relevant_keys = self._get_relevant_keys(start_time, end_time, min_lat, max_lat, min_lon, max_lon)
        sub_dfs = []
        if len(relevant_keys) > 0:
            for rkey in relevant_keys:
                if self.in_memory:
                    df = self.loaded_catalogs[rkey]
                else:
                    df = pd.read_csv(file, parse_dates=[self.time_name])

                sub_df = df[(df[self.time_name] >= start_time) & (df[self.time_name] < end_time)]
                sub_df = sub_df[(sub_df[self.latitude_name] >= min_lat) & (sub_df[self.latitude_name] <= max_lat)]
                sub_df = sub_df[(sub_df[self.longitude_name] >= min_lon) & (sub_df[self.longitude_name] < max_lon)]
                if len(sub_df) > 0:
                    sub_dfs.append(sub_df)
            sub_dfs = pd.concat(sub_dfs)
            return sub_dfs
        else:
            return None

    def __len__(self):
        return self.cat_metadatas['num_events'].sum()

In [7]:
mc_dataset = MultiCatalogDataset(root_dir="catalogs/isc/")

In [8]:
#mc_dataset._get_relevant_keys("2000-01-01", "2015-01-01", 0, 50, 0, 60)

In [9]:
mc_dataset._load_data("2000-01-01", "2010-01-01", 0, 50, 0, 60)

,event_id,time,latitude,longitude,depth,magnitude,magnitude_type,magnitude_family
0,smi:ISC/evid=1819247,2001-01-01 19:12:21.060,43.1596,17.9308,4.5688,3.1,ML,ML
1,smi:ISC/evid=3398771,2001-01-02 22:53:25.600,39.3700,19.5400,10.0000,3.0,MD,unspecified
2,smi:ISC/evid=1819249,2001-01-03 08:04:28.270,42.9990,17.8940,12.9000,3.3,ML,ML
3,smi:ISC/evid=1834188,2001-01-03 10:34:42.110,36.5316,4.9497,0.0000,4.5,mb,mb
4,smi:ISC/evid=1834196,2001-01-03 22:34:33.900,43.6350,11.2670,2.6000,2.2,ML,ML
...,...,...,...,...,...,...,...,...
2722,smi:ISC/evid=2978795,2000-12-31 01:27:26.700,35.4400,24.4100,14.0000,3.5,MD,unspecified
2723,smi:ISC/evid=1769179,2000-12-31 02:23:33.800,38.3240,31.5520,10.0000,3.0,MD,unspecified
2724,smi:ISC/evid=1769185,2000-12-31 03:28:23.200,37.8660,31.6770,10.0000,3.1,MD,unspecified
2725,smi:ISC/evid=1769191,2000-12-31 09:35:40.500,37.6380,30.8510,10.0000,3.0,MD,unspecified


In [17]:
import torch
import torch.nn as nn

class LatLonTimeEmbedding(nn.Module):
    def __init__(
        self,
        hidden_dim,
        emb_dim,
        origin_year=1970,
        year_scale=100.0,
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.emb_dim = emb_dim
        self.origin_year = origin_year
        self.year_scale = year_scale

        # second, minute, hour, day_of_year, month:
        # each -> [normalized, sin, cos]
        # year -> [relative year]
        self.num_time_features = 5 * 3 + 1  # 16

        self.space_emb = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.SiLU(),
        )

        self.time_emb = nn.Sequential(
            nn.Linear(self.num_time_features, hidden_dim),
            nn.SiLU(),
        )

        self.spacetime_emb = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, emb_dim),
        )

    @staticmethod
    def embed_periodic(x, period):
        """
        x: (...)
        Returns (..., 3):
            normalized coordinate, sin phase, cos phase
        """
        x_norm = x / period
        phase = 2.0 * torch.pi * x_norm

        return torch.stack(
            [
                x_norm,
                torch.sin(phase),
                torch.cos(phase),
            ],
            dim=-1,
        )

    def embed_latlon(self, lat_lons):
        """
        lat_lons: (..., 2), in degrees
        """
        if lat_lons.shape[-1] != 2:
            raise ValueError("lat_lons must have shape (..., 2)")

        lat = torch.deg2rad(lat_lons[..., 0])
        lon = torch.deg2rad(lat_lons[..., 1])

        cos_lat = torch.cos(lat)

        xyz = torch.stack(
            [
                cos_lat * torch.cos(lon),
                cos_lat * torch.sin(lon),
                torch.sin(lat),
            ],
            dim=-1,
        )

        return self.space_emb(xyz)

    def embed_time(self, times):
        """
        times: (..., 6)

        Last dimension:
            [second, minute, hour, day_of_year, month, year]

        Conventions:
            second      : 0..59 (+ fractional seconds allowed)
            minute      : 0..59
            hour        : 0..23
            day_of_year : 0..364/365
            month       : 0..11
            year        : e.g. 2026
        """

        if times.shape[-1] != 6:
            raise ValueError(
                "times must have shape (..., 6): "
                "[second, minute, hour, day_of_year, month, year]"
            )

        second = times[..., 0]
        minute = times[..., 1]
        hour = times[..., 2]
        day = times[..., 3]
        month = times[..., 4]
        year = times[..., 5]

        # Exact number of days in this year
        year_int = year.long()

        is_leap = (
            ((year_int % 4 == 0) & (year_int % 100 != 0))
            | (year_int % 400 == 0)
        )

        days_in_year = torch.where(
            is_leap,
            torch.tensor(366.0, device=times.device, dtype=times.dtype),
            torch.tensor(365.0, device=times.device, dtype=times.dtype),
        )

        e_second = self.embed_periodic(second, 60.0)
        e_minute = self.embed_periodic(minute, 60.0)
        e_hour = self.embed_periodic(hour, 24.0)

        # Need variable period because of leap years
        day_norm = day / days_in_year
        day_phase = 2.0 * torch.pi * day_norm
        e_day = torch.stack(
            [
                day_norm,
                torch.sin(day_phase),
                torch.cos(day_phase),
            ],
            dim=-1,
        )

        e_month = self.embed_periodic(month, 12.0)

        # Year is NOT periodic
        e_year = (
            (year - self.origin_year) / self.year_scale
        ).unsqueeze(-1)

        time_features = torch.cat(
            [
                e_second,
                e_minute,
                e_hour,
                e_day,
                e_month,
                e_year,
            ],
            dim=-1,
        )

        return self.time_emb(time_features)

    def forward(self, lat_lons, times):
        space_emb = self.embed_latlon(lat_lons)
        time_emb = self.embed_time(times)

        spacetime = torch.cat(
            [space_emb, time_emb],
            dim=-1,
        )

        return self.spacetime_emb(spacetime)

In [18]:
def datetime64_to_times(datetimes):
    """
    Convert np.datetime64 scalar/array to:
        [..., 6] = [second, minute, hour, day_of_year, month, year]

    Conventions:
        second      : [0, 60), fractional seconds preserved
        minute      : 0..59
        hour        : 0..23
        day_of_year : 0..364/365
        month       : 0..11
        year        : e.g. 2026
    """
    dt = np.asarray(datetimes).astype("datetime64[ns]")

    year = dt.astype("datetime64[Y]")
    month = dt.astype("datetime64[M]")
    day = dt.astype("datetime64[D]")
    hour = dt.astype("datetime64[h]")
    minute = dt.astype("datetime64[m]")

    years = year.astype(np.int64) + 1970

    months = (
        month.astype(np.int64)
        - year.astype("datetime64[M]").astype(np.int64)
    )

    days_of_year = (
        day - year.astype("datetime64[D]")
    ).astype("timedelta64[D]").astype(np.int64)

    hours = (
        hour - day.astype("datetime64[h]")
    ).astype("timedelta64[h]").astype(np.int64)

    minutes = (
        minute - hour.astype("datetime64[m]")
    ).astype("timedelta64[m]").astype(np.int64)

    # Fractional seconds preserved
    seconds = (
        (dt - minute.astype("datetime64[ns]"))
        / np.timedelta64(1, "s")
    )

    return np.stack(
        [
            seconds,
            minutes,
            hours,
            days_of_year,
            months,
            years,
        ],
        axis=-1,
    ).astype(np.float32)

In [ ]:
class MultiscaleMemory(nn.Module):
    def __init__(self, event_dim, memory_dim, num_scales):
        super().__init__()

        self.num_scales = num_scales
        self.memory_dim = memory_dim

        # One decay timescale per memory
        self.log_tau = nn.Parameter(
            torch.zeros(num_scales)
        )

        # shared block
        self.event_encoder = nn.Sequential(
            nn.Linear(event_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
        )

        # independent memory for each timescale
        self.writers = nn.ModuleList([
            nn.Linear(128, memory_dim)
            for _ in range(num_scales)
        ])

    def decay(self, memories, dt):
        """
        memories:
            (..., num_scales, memory_dim)

        dt:
            (...) elapsed time
        """

        tau = torch.exp(self.log_tau)

        decay = torch.exp(
            -dt.unsqueeze(-1) / tau
        )

        return memories * decay.unsqueeze(-1)


    def update(self, memories, event_features):
        u = self.event_encoder(event_features)

        jumps = torch.stack(
            [writer(u) for writer in self.writers],
            dim=-2,
        )

        return memories + jumps


    def forward(self, memories, dt, event_features):
        memories_minus = self.decay(memories, dt)
        memories_plus = self.update(
            memories_minus,
            event_features,
        )

        return memories_minus, memories_plus
        